(sec:md)=
# Molecular dynamics
Molecular dynamics (MD) simulates the time evolution of a molecular system by numerically integrating Newton's equations of motion. At each timestep, forces acting on every atom are derived from the potential energy function of the underlying force field, and atomic positions and velocities are updated accordingly. By propagating the system over many timesteps, MD generates a trajectory from which thermodynamic quantities (temperature, pressure, free energy) and structural properties (radial distribution functions, conformational populations) can be extracted via statistical mechanics.

VeloxChem provides the `OpenMMDynamics` class as a high-level interface to run MD simulations with use of the [OpenMM](https://openmm.org) engine, see {cite}`vlx_workflow`. Starting from a force field topology prepared with the `MMForceFieldGenerator`, a simulation can be set up and executed in a few lines of Python. 

## Creating solute force field

In [1]:
import veloxchem as vlx

molecule = vlx.Molecule.read_name("ethanol")

ff_gen = vlx.MMForceFieldGenerator()
ff_gen.create_topology(molecule)

Reading ethanol from PubChem...

Reference: S. Kim, J. Chen, T. Cheng, A. Gindulyte, J. He, S. He, Q. Li, B. A. Shoemaker, P. A. Thiessen, B. Yu, L. Zaslavsky, J. Zhang, E. E. Bolton, Nucleic Acids Res., 2025, 53, D1516-D1525.

Please double-check the compound since names may refer to more than one record.

* Info * Using 6-31G* basis set for RESP charges...                                                                       
* Info * Sum of partial charges is not a whole number.                                                                    
* Info * Compensating by removing 1.000e-06 from the largest charge.                                                      
                                                                                                                          
* Info * Using GAFF (v2.11) parameters.                                                                                   
         Reference: J. Wang, R. M. Wolf, J. W. Caldwell, P. A. Kollman, D. A

(sec:MD-run)=
## Running MD simulation

You can directly solvate your system by using the solvent option when you create a system. For more information about solvation, please visit the [system solvation](#sec:xc-functionals) page.

In [2]:
opm_dyn = vlx.OpenMMDynamics()

opm_dyn.create_system_from_molecule(
    molecule,
    ff_gen,
    filename="ethanol",  # system parameters and coordinates written to .xml and .pdb
    solvent="tip3p",
    residue_name="ETH",
)

opm_dyn.run_md(
    ensemble="NVT",
    temperature=300,  # in Kelvin
    timestep=2.0,  # in fs
    nsteps=10000,
    snapshots=1000,
    traj_file="ethanol_md.pdb",
)

                                               VeloxChem Solvation Builder                                                
                                                                                                                          
* Info * Solvating the solute with tip3p molecules                                                                        
* Info * Padding: 1.0 nm                                                                                                  
                                                                                                                          
* Info * NPT Equilibration of the box requested                                                                           
                                                                                                                          
* Info * The box size is: 2.30 x 2.30 x 2.30 nm^3                                                                         
* Info * The vol

In [ ]:
import py3Dmol as p3d

with open("../output_files/ethanol_md.pdb", "r") as f:
    pdb_data = f.read()

view = p3d.view(width=600, height=450)
view.addModelsAsFrames(pdb_data, "pdb", {"keepH": True})
view.setStyle({"resn": "MOL"}, {"stick": {"radius": 0.15}, "sphere": {"scale": 0.25}})
view.setStyle({"resn": "S01"}, {"stick": {"radius": 0.20}})
view.animate({"loop": "forward", "reps": 0})
view.zoomTo()
view.show()